### LLM Chain 만들기


## 1. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add python-dotenv langchain langchain-openai

### 2) GROQ 인증키 설정

In [1]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:3])

gsk


In [2]:
import langchain
print(langchain.__version__)

1.4.3


In [3]:
from langchain_openai import ChatOpenAI

# Groq(OpenAI 호환) 공통 설정 — 모델/온도 변경은 이곳만 수정
GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트
GROQ_MODEL = "openai/gpt-oss-120b"  # 대안: "qwen/qwen3.8-27b"
#MOVIE_MODEL = "qwen/qwen3.8-27b"  # 영화 추천 예제용 
TEMPERATURE = 0.7


def get_llm(model: str = GROQ_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    """Groq 엔드포인트를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        model=model,
        temperature=temperature,
    )


# 기본 llm (이후 셀에서 재사용)
llm = get_llm()

## 2. LLM Chain

### 1) Prompt + LLM


In [4]:
# model: 공통 설정 셀의 llm 사용

# chain 실행
result = llm.invoke("인공지능 모델의 학습 원리에 대하여 쉽게 설명해 주세요.")
print(type(result))
print(result)

<class 'langchain_core.messages.ai.AIMessage'>
content='## 인공지능 모델, 특히 **머신러닝·딥러닝**이 어떻게 배우는지 쉽게 풀어볼게요  \n\n---\n\n### 1. “학생”과 “문제”를 만든다\n| 요소 | 비유 | 설명 |\n|------|------|------|\n| **데이터** | 교과서·연습문제 | 모델에게 보여줄 입력‑출력 쌍. 예) 사진(입력) → “고양이/강아지”(정답) |\n| **모델** | 학생 | 입력을 받아서 답을 예측하는 **함수**(수학식). 파라미터(가중치·편향)는 학생의 “지식”에 해당. |\n| **목표** | 시험 점수 | 정답과 예측이 얼마나 차이 나는지 나타내는 **손실(Loss) 함수**. 차이가 크면 점수가 낮다. |\n\n---\n\n### 2. “공부”는 **파라미터를 조정**하는 과정\n1. **예측**  \n   - 현재 파라미터(학생의 지식)로 입력을 넣어 예측값을 만든다.  \n2. **오답 확인**  \n   - 손실 함수를 써서 예측값과 정답 사이의 차이를 계산한다.  \n3. **수정(학습)**  \n   - 차이를 줄이기 위해 파라미터를 조금씩 바꾼다.  \n   - 이때 쓰는 대표적인 방법이 **경사하강법(Gradient Descent)** → “오답이 가장 크게 나는 방향으로 조금씩 뒤로 물러난다” 라고 생각하면 됨.  \n\n이 과정을 **데이터 전체(또는 작은 배치)마다 반복**하면 파라미터가 점점 최적에 가까워져서 손실이 작아지고, 모델이 더 정확해진다.\n\n---\n\n### 3. “학습”이 끝나는 시점\n| 상황 | 설명 |\n|------|------|\n| **에포크(Epoch) 수** | 전체 데이터셋을 몇 번 다 보았는지. 보통 10~100번 정도. |\n| **손실 감소가 멈춤** | 더 이상 크게 개선되지 않을 때. |\n| **검증 성능** | 훈련 데이터가 아니라 별도로 남겨둔 검증 데이터에서 정확도가 최고가 되는 시점

In [5]:
print(result.content)

## 인공지능 모델, 특히 **머신러닝·딥러닝**이 어떻게 배우는지 쉽게 풀어볼게요  

---

### 1. “학생”과 “문제”를 만든다
| 요소 | 비유 | 설명 |
|------|------|------|
| **데이터** | 교과서·연습문제 | 모델에게 보여줄 입력‑출력 쌍. 예) 사진(입력) → “고양이/강아지”(정답) |
| **모델** | 학생 | 입력을 받아서 답을 예측하는 **함수**(수학식). 파라미터(가중치·편향)는 학생의 “지식”에 해당. |
| **목표** | 시험 점수 | 정답과 예측이 얼마나 차이 나는지 나타내는 **손실(Loss) 함수**. 차이가 크면 점수가 낮다. |

---

### 2. “공부”는 **파라미터를 조정**하는 과정
1. **예측**  
   - 현재 파라미터(학생의 지식)로 입력을 넣어 예측값을 만든다.  
2. **오답 확인**  
   - 손실 함수를 써서 예측값과 정답 사이의 차이를 계산한다.  
3. **수정(학습)**  
   - 차이를 줄이기 위해 파라미터를 조금씩 바꾼다.  
   - 이때 쓰는 대표적인 방법이 **경사하강법(Gradient Descent)** → “오답이 가장 크게 나는 방향으로 조금씩 뒤로 물러난다” 라고 생각하면 됨.  

이 과정을 **데이터 전체(또는 작은 배치)마다 반복**하면 파라미터가 점점 최적에 가까워져서 손실이 작아지고, 모델이 더 정확해진다.

---

### 3. “학습”이 끝나는 시점
| 상황 | 설명 |
|------|------|
| **에포크(Epoch) 수** | 전체 데이터셋을 몇 번 다 보았는지. 보통 10~100번 정도. |
| **손실 감소가 멈춤** | 더 이상 크게 개선되지 않을 때. |
| **검증 성능** | 훈련 데이터가 아니라 별도로 남겨둔 검증 데이터에서 정확도가 최고가 되는 시점. |

---

### 4. 흔히 겪는 문제와 해결법 (쉽게 이해할 수 있는 비유)

| 문제 | 비유 | 해결법 |
|------|------|-----

### 2) PromptTemplate + LLM

In [6]:
print(f'model name = {llm.model_name}')

model name = openai/gpt-oss-120b


In [7]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("You are an expert in AI Expert. Answer the question.\
                                       <Question>: {input}에 대해 쉽게 설명해주세요.")

print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.prompt.PromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} template='You are an expert in AI Expert. Answer the question.                                       <Question>: {input}에 대해 쉽게 설명해주세요.'


In [8]:
# llm: 공통 설정 셀의 llm 사용
# chain 연결 (LCEL)
chain = prompt | llm
print(type(chain))

# chain 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.ai.AIMessage'>
content='### 인공지능 모델, 특히 **딥러닝(신경망)** 이 어떻게 “학습”하는지 쉽게 풀어볼게요.\n\n---\n\n## 1️⃣ 학습의 기본 흐름\n\n| 단계 | 무슨 일? | 비유 |\n|------|----------|------|\n| **데이터 준비** | 모델에게 보여줄 입력‑출력 쌍을 모아요. (예: 사진‑고양이/강아지 라벨) | 선생님이 학생에게 문제와 정답을 나눠 주는 것 |\n| **예측(Forward)** | 입력을 모델에 넣어 **예측값**을 만든다. | 학생이 문제를 풀고 답을 적는 순간 |\n| **오차 계산(손실 함수)** | 예측값과 정답 사이의 차이를 숫자로 측정한다. | 정답과 얼마나 틀렸는지 “점수”를 매기는 것 |\n| **피드백(Backpropagation)** | 오차가 모델 안의 **가중치**에 어떻게 영향을 주는지 역으로 전파한다. | 틀린 이유를 찾아서 “어디를 고쳐야 할까?”를 알려주는 선생님 |\n| **가중치 업데이트(Gradient Descent)** | 가중치를 조금씩 바꿔서 오차를 줄인다. | 학생이 틀린 문제를 다시 풀면서 답을 고쳐 나가는 과정 |\n| **반복(Epoch)** | 위 과정을 전체 데이터에 여러 번 반복한다. | 여러 번 연습문제를 풀며 실력을 키우는 것 |\n\n---\n\n## 2️⃣ 핵심 개념을 일상적인 비유로\n\n### 2‑1. 가중치 = “조리법”  \n신경망 안에는 수많은 **가중치**가 있어요. 이것은 “재료를 얼마나 넣을지”를 결정하는 **조리법**과 비슷합니다.  \n- 가중치가 크면 해당 입력 신호가 강하게 전달되고, 작으면 약하게 전달돼요.  \n- 학습 과정은 이 조리법을 **점점 더 맛있는(정확한) 요리**가 되도록 바꾸는 일입니다.\n\n### 2‑2. 손실 함수 

In [9]:
print(result.content)

### 인공지능 모델, 특히 **딥러닝(신경망)** 이 어떻게 “학습”하는지 쉽게 풀어볼게요.

---

## 1️⃣ 학습의 기본 흐름

| 단계 | 무슨 일? | 비유 |
|------|----------|------|
| **데이터 준비** | 모델에게 보여줄 입력‑출력 쌍을 모아요. (예: 사진‑고양이/강아지 라벨) | 선생님이 학생에게 문제와 정답을 나눠 주는 것 |
| **예측(Forward)** | 입력을 모델에 넣어 **예측값**을 만든다. | 학생이 문제를 풀고 답을 적는 순간 |
| **오차 계산(손실 함수)** | 예측값과 정답 사이의 차이를 숫자로 측정한다. | 정답과 얼마나 틀렸는지 “점수”를 매기는 것 |
| **피드백(Backpropagation)** | 오차가 모델 안의 **가중치**에 어떻게 영향을 주는지 역으로 전파한다. | 틀린 이유를 찾아서 “어디를 고쳐야 할까?”를 알려주는 선생님 |
| **가중치 업데이트(Gradient Descent)** | 가중치를 조금씩 바꿔서 오차를 줄인다. | 학생이 틀린 문제를 다시 풀면서 답을 고쳐 나가는 과정 |
| **반복(Epoch)** | 위 과정을 전체 데이터에 여러 번 반복한다. | 여러 번 연습문제를 풀며 실력을 키우는 것 |

---

## 2️⃣ 핵심 개념을 일상적인 비유로

### 2‑1. 가중치 = “조리법”  
신경망 안에는 수많은 **가중치**가 있어요. 이것은 “재료를 얼마나 넣을지”를 결정하는 **조리법**과 비슷합니다.  
- 가중치가 크면 해당 입력 신호가 강하게 전달되고, 작으면 약하게 전달돼요.  
- 학습 과정은 이 조리법을 **점점 더 맛있는(정확한) 요리**가 되도록 바꾸는 일입니다.

### 2‑2. 손실 함수 = “점수표”  
예측이 정답과 얼마나 다른지를 수치화한 것이 **손실(Loss)** 입니다.  
- 손실이 작을수록 “점수표”에 높은 점수를 받은 것이죠.  
- 목표는 손실을 **가능한 한 0에 가깝게** 만드는 겁니다.

### 2‑3. 경

### 3) PromptTemplate + LLM(invoke()) + StrOutputParser

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

output_parser = StrOutputParser()

# 2. chain 생성 (LCEL)
chain = prompt | llm | output_parser
print(type(chain))

# 3. chain의 invoke 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.base.TextAccessor'>
### 인공지능 모델이 어떻게 배우는지 쉽게 풀어볼게요  

---

## 1. 기본 아이디어: “반복 훈련 + 피드백”

인공지능, 특히 **머신러닝·딥러닝 모델**은 **데이터**와 **목표(정답)**를 가지고 “시도 → 평가 → 수정”을 여러 번 반복하면서 스스로 규칙을 찾아갑니다.  

> **비유** : 아이가 퍼즐을 맞출 때, 처음엔 조각을 무작위로 끼워보고 “이건 안 맞아!” 라는 피드백을 받습니다. 그 피드백을 바탕으로 다음엔 더 좋은 조각을 골라 넣는 과정을 계속하면 결국 퍼즐을 완성하죠. AI도 비슷합니다.

---

## 2. 핵심 요소 3가지

| 요소 | 역할 | 쉬운 설명 |
|------|------|-----------|
| **데이터** | 배울 재료 | 사진, 텍스트, 소리 등 “입력”과 그에 맞는 “정답(라벨)”을 모아 둔 것 |
| **모델(네트워크)** | 규칙을 만들 기계 | 뇌의 뉴런처럼 연결된 작은 단위(노드)들이 모여서 입력을 변환해 결과를 내는 구조 |
| **손실 함수(Loss)** | 얼마나 틀렸는지 알려주는 점수 | 정답과 모델이 낸 예측 사이의 차이를 숫자로 나타냅니다. 점수가 클수록 “아직 많이 틀렸어” 라는 의미 |

---

## 3. 학습 과정: “경사 하강법(Gradient Descent)”

1. **예측**  
   - 모델에 입력 데이터를 넣어 예측값을 만든다. (예: 사진 → 고양이/강아지 중 어느 것인가?)

2. **평가 (손실 계산)**  
   - 예측값과 실제 정답을 비교해 손실을 구한다.  
   - 손실이 작을수록 모델이 잘 맞춘 것이고, 크면 틀렸다는 뜻.

3. **수정 (가중치 업데이트)**  
   - 손실을 최소화하도록 모델 안에 있는 **가중치(연결 강도)** 를 조금씩 바꾼다

### 4) PromptTemplate + LLM(stream()) + StrOutputParser

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

# chain 연결 (LCEL)
chain = prompt | llm | StrOutputParser()

# 스트리밍 출력을 위한 요청
answer = chain.stream({"input": "인공지능 모델의 학습 원리"})
# 스트리밍 출력
#print(answer)

for token in answer:
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

## 인공지능 모델은 어떻게 배우나요? – 쉽게 풀어쓴 이야기

### 1. “학생”과 “교재”를 준비한다
| 요소 | AI 용어 | 의미 |
|------|---------|------|
| **교재** | 데이터 (데이터셋) | 모델이 배워야 할 예시들. 사진, 텍스트, 소리 등 |
| **학생** | 모델 (신경망) | 입력을 받아서 “정답”을 예측하는 프로그램 |
| **정답** | 라벨 (정답 레이블) | 교재에 적힌 정답. 예) 사진 → “고양이”, “개” 등 |

### 2. “문제 풀고 채점받기” – **전방 전달 (Forward Pass)**
1. **입력**을 모델에 넣는다. (예: 고양이 사진)
2. 모델은 내부의 **가중치**와 **편향**이라는 숫자를 이용해 **출력**을 만든다. (예: “고양이 0.8, 개 0.2”)
3. 이 출력값을 **예측**이라고 부른다.

### 3. “오답을 확인하고 교정하기” – **손실 함수 (Loss)와 역전파 (Back‑propagation)**
1. **손실 함수**는 “예측이 정답과 얼마나 차이가 나는가?”를 수치화한다.  
   - 예) 정답이 “고양이”인데 모델이 0.8를 줬다면 손실은 작고, 0.2라면 손실은 큼.
2. 손실이 **큰** 경우, 모델이 **잘못된 방향**으로 가중치를 사용하고 있다는 신호.
3. **역전파**는 “어디를 고쳐야 할까?”를 계산한다.  
   - 수학적으로는 **미분(gradient)**을 이용해 각 가중치가 손실에 얼마나 영향을 주는지 알아낸다.

### 4. “가중치를 조금씩 바꾸며 공부하기” – **경사 하강법 (Gradient Descent)**
- 구한 미분값(gradient)을 이용해 가중치를 **조금씩** 업데이트한다.  
  \[
  \text{새 가중치} = \text{현재 가중치} - \eta \times \text{gradient}
  \]
- 여기서 **η (학습률)** 은 “한 번에 얼마나 많이 바꿀까?”를 정한다. 너무 크게 하면 넘어가고

##### 2) Multiple Chains
* Multi Chain을 활용한 영화 추천 및 줄거리 요약 

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고,\
                                            줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요")

# 영화 예제용 모델 (공통 설정 셀의 get_llm 사용)
llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
movie = chain1.invoke({"genre": "Drama"})  # 영화 제목 얻기

print(type(movie))
print(" ===> 추천된 영화:")  # movie 값 출력
print(movie)


<class 'langchain_core.messages.base.TextAccessor'>
 ===> 추천된 영화:
**추천 영화: <그린 마일 (The Green Mile, 1999)**  

- **감독**: 프랭크 다라본트  
- **주연**: 톰 행크스, 마이클 클라크·던컨, 데이비드 모스  
- **줄거리**: 1930년대 미국 남부 교도소에서 사형수들을 담당하던 교도관 폴(톰 행크스)과, 초능력을 지닌 신비로운 수감자 존 코피(마이클 클라크·던컨)의 이야기를 그립니다. 인간성, 정의, 사랑, 그리고 구원의 감동을 섬세하게 풀어낸 드라마입니다.  
- **추천 이유**  
  1. **감동적인 인간 드라마** – 억압된 환경 속에서도 피어나는 인간의 선함과 연민을 깊이 있게 탐구합니다.  
  2. **뛰어난 연기** – 톰 행크스와 마이클 클라크·던컨의 섬세하고 진솔한 연기가 관객을 사로잡습니다.  
  3. **시각·음악적 완성도** – 고전적인 흑백 톤과 존 윅스의 서정적인 음악이 분위기를 한층 고조시킵니다.  
  4. **보편적인 메시지** – “우리 모두는 서로에게 어떤 ‘그린 마일’이 될 수 있다”는 인간애와 용서의 메시지는 시대와 국경을 초월합니다.  

**한 줄 요약**: 감동적인 스토리와 명연기로 가득한 ‘그린 마일’은 드라마 장르를 사랑하는 이라면 반드시 봐야 할 작품입니다. 즐거운 감상 되세요!


In [13]:
# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)
print(chain2)

# 실행: "SF" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:\n", response) 

first={
  movie: ChatPromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, template='{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.'), additional_kwargs={})])
         | ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000263E463B530>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000263E463B260>, root_client=<openai.OpenAI object at 0x00000263CCC23EC0>, root_async_client=<openai.AsyncOpenAI object at 0x00000263E4508B90>, model_name='openai/gpt-oss-120b', temperature=0.7, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.groq.com/openai/v1', stream_chunk_timeout=120.0)
         | StrOutputParser()
} middle=[ChatPromptTemplate

##### chain1과 chain2에서 영화 제목이 일관되게 전달 되도록 변경 

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from pprint import pprint

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고, 줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요.")

llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: "Drama" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:")
pprint(response)


🔹 영화 줄거리 요약:
('**그린 북 (Green Book, 2018)**  \n'
 '\n'
 '---\n'
 '**제목**: 그린 북 (Green Book)  \n'
 '**감독**: 피터 패럴리 (Peter Farrelly)  \n'
 '**캐스팅**: 마하샬라 알리 (Mahershala Ali) – 돈 샤프 (Don\u202fSharp) / 비고 모텐슨 (Viggo '
 'Mortensen) – 토니 리프 (Tony\u202fLip) / 네이선 킬리언 (Nicole Kidman) – 마리안 샤프 '
 '(Marian\u202fSharp) 등  \n'
 '**줄거리**: 1962년, 세계적인 클래식 피아니스트 돈 샤프는 남부 투어를 위해 전직 폭력배였던 토니 리프를 운전사 겸 보디가드로 '
 '고용한다. 인종 차별이 일상인 미국 남부를 여행하며 두 사람은 문화적 갈등과 오해를 겪지만, 서로의 삶을 이해하고 진정한 ‘형제’ 같은 '
 '우정을 쌓아간다. 영화는 그들의 여정을 통해 인간성, 용서, 그리고 서로 다른 배경을 뛰어넘는 연대의 가치를 따뜻한 유머와 감동으로 '
 '그려낸다.')


In [15]:
# 모델 설정 (공통 설정 셀의 get_llm 사용)
model = get_llm(GROQ_MODEL)

# 멀티 체인 실행 함수
def movie_chain(genre):
    print(f"\n🔹 1단계: '{genre}' 장르 영화 추천")
    
    # 체인 1: 영화 추천
    response1 = model.invoke(f"{genre} 장르에서 유명한 영화 제목 하나만 알려줘")
    movie = response1.content.strip()
    print(f"   → 추천 영화: {movie}")
    
    print(f"\n 2단계: '{movie}' 영화 정보 조회")
    
    # 체인 2: 영화 정보
    response2 = model.invoke(f"{movie} 영화에 대해 간단히 알려줘. 줄거리, 감독, 출연진 포함해서")
    info = response2.content
    print(f"   → 영화 정보:\n{info}")
    
    return movie, info

# 실행
genre = input("원하는 장르를 입력하세요: ")
movie, info = movie_chain(genre)
print(f"\n 최종 결과: '{movie}' 영화가 추천되었습니다!")


🔹 1단계: '액션' 장르 영화 추천
   → 추천 영화: "다이 하드 (Die Hard)"는 액션 장르에서 가장 유명하고 사랑받는 영화 중 하나입니다.

 2단계: '"다이 하드 (Die Hard)"는 액션 장르에서 가장 유명하고 사랑받는 영화 중 하나입니다.' 영화 정보 조회
   → 영화 정보:
**다이 하드 (Die Hard, 1988)**  

| 항목 | 내용 |
|------|------|
| **감독** | 존 맥티어넌 (John McTiernan) |
| **주연** | 브루스 윌리스 (John McClane 역) <br> Alan Rickman (Hans Gruber 역) <br> Bonnie Bedelia (Holly McClane 역) <br> Reginald VelJohnson (Al Powell 역) <br> William Atherton (Harry Lambert 역) |
| **제작·배급** | 20th Century Fox (배급) |
| **개봉** | 1988년 7월 15일 (미국) |
| **장르** | 액션·스릴러·범죄 |

---

### 간단한 줄거리

크리스마스 이브, 뉴욕 경찰관 **존 맥클레인(브루스 윌리스)** 은 LA에 있는 아내 **홀리**와 동료들을 만나기 위해 나카토미 플라자(가상의 고층 빌딩)로 출장 온다.  

그곳에서 **한스 그루버(앨런 리크먼)** 일당이 고도로 조직된 은행 강도를 가장해 건물 전체를 점거한다. 인질을 잡고, 금고를 털려는 목적이었지만, 실제로는 **미국 연방 준비제도(Federal Reserve)** 의 금고에 있는 4억 달러를 훔치려는 대규모 절도 작전이다.  

맥클레인은 건물 안에 갇힌 채 **전기와 통신 설비를 이용해 혼자서** 테러리스트들을 하나씩 무력화한다. 그는 라디오와 경찰 무전, 심지어는 건물 내의 소방 호스와 엘리베이터를 활용해 저항한다.  

극중 내내 **“Yippee‑ki‑yay, motherf***er!”** 라는 대사와 함께, 맥클레인은 물리적·심리적